# Generate BGE-M3 450-token chunk embeddings

Create article-bound chunks with a structure-preserving recursive splitter and generate the production BGE-M3 embeddings. The splitter tries paragraph breaks, newlines, Khmer sentence punctuation, spaces, and finally character boundaries.

In [ ]:
import gc
import json
import logging
import re
from pathlib import Path

import numpy as np
from transformers import AutoTokenizer

logging.getLogger('torch.distributed.elastic.multiprocessing.redirects').setLevel(logging.ERROR)
try:
    from transformers.utils import logging as transformers_logging
    transformers_logging.disable_progress_bar()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / 'data-pipeline').is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data-pipeline' / 'data'
INPUT_PATH = DATA_DIR / 'processed' / 'articles_normalized.jsonl'
CHUNK_PATH = DATA_DIR / 'processed' / 'article_chunks_bge450.jsonl'
OUTPUT_DIR = DATA_DIR / 'processed' / 'embeddings'
TOKENIZER_NAME = 'aisingapore/SEA-LION-E5-Embedding-600M'
TOKENIZER = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
TOKENIZER.model_max_length = 10**9
MAX_TOKENS = 450
MIN_CHUNK_CHARS = 80
MODELS = {
    'bge-m3': {'model': 'BAAI/bge-m3', 'provider': 'sentence-transformer', 'batch_size': 2, 'output': 'chunks_bge_m3_recursive_450_tokens'},
    }

def token_count(text):
    return len(TOKENIZER.encode(text, add_special_tokens=False))

def context_prefix(article):
    values = [article.get('document_title'), f"មាត្រា {article.get('article_number')}" if article.get('article_number') is not None else None, article.get('chapter_title'), article.get('section_title')]
    return ' | '.join(str(value).strip() for value in values if value)

def pack_units(units, budget):
    chunks, current = [], ''
    for unit in units:
        unit = unit.strip()
        if not unit:
            continue
        candidate = f'{current} {unit}'.strip() if current else unit
        if current and token_count(candidate) > budget:
            chunks.append(current)
            current = unit
        else:
            current = candidate
    if current:
        chunks.append(current)
    return chunks

def recursive_token_split(text, budget, level=0):
    text = text.strip()
    if not text:
        return []
    if token_count(text) <= budget:
        return [text]
    separators = [r'\n\n+', r'\n+', r'(?<=[។៕?!])\s+', '\u200b', r'\s+']
    if level < len(separators):
        units = [unit.strip() for unit in re.split(separators[level], text) if unit.strip()]
        if len(units) > 1:
            chunks = pack_units(units, budget)
            if all(token_count(chunk) <= budget for chunk in chunks):
                return chunks
            return [piece for chunk in chunks for piece in recursive_token_split(chunk, budget, level + 1)]
        return recursive_token_split(text, budget, level + 1)
    pieces, current = [], ''
    for character in text:
        candidate = current + character
        if current and token_count(candidate) > budget:
            pieces.append(current)
            current = character
        else:
            current = candidate
    if current:
        pieces.append(current)
    return pieces

def split_article(text, prefix=''):
    text = ' '.join(str(text).split())
    budget = max(1, MAX_TOKENS - token_count(prefix))
    chunks = recursive_token_split(text, budget)
    if len(chunks) > 1 and len(chunks[-1]) < MIN_CHUNK_CHARS:
        candidate = f'{chunks[-2]} {chunks[-1]}'.strip()
        if token_count(candidate) <= budget:
            chunks[-2], chunks[-1] = candidate, None
            chunks = [chunk for chunk in chunks if chunk]
    return chunks

def embedding_text(article, chunk_text):
    context = [article.get('document_title'), f"មាត្រា {article.get('article_number')}" if article.get('article_number') is not None else None, article.get('chapter_title'), article.get('section_title')]
    context = ' | '.join(str(value).strip() for value in context if value)
    return f'{context}\n{chunk_text}'.strip() if context else chunk_text

def create_chunks():
    records = [json.loads(line) for line in INPUT_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
    output = []
    for article in records:
        prefix = context_prefix(article)
        chunks = split_article(article.get('search_text', article.get('text', '')), prefix)
        for index, text in enumerate(chunks, start=1):
            output.append({'chunk_id': f"{article['id']}-chunk-{index}", 'parent_id': article['id'], 'article_id': article['article_id'], 'document_id': article['document_id'], 'article_number': article['article_number'], 'chunk_index': index, 'chunk_count': len(chunks), 'text': text, 'search_text': text, 'document_title': article.get('document_title'), 'document_number': article.get('document_number'), 'document_type': article.get('document_type'), 'source_url': article.get('source_url'), 'pdf_url': article.get('pdf_url'), 'chapter_number': article.get('chapter_number'), 'chapter_title': article.get('chapter_title'), 'section_number': article.get('section_number'), 'section_title': article.get('section_title')})
    for record in output:
        record['search_text'] = embedding_text(record, record['text'])
    CHUNK_PATH.parent.mkdir(parents=True, exist_ok=True)
    CHUNK_PATH.write_text(''.join(json.dumps(record, ensure_ascii=False) + '\n' for record in output), encoding='utf-8')
    return output

def batches(values, size):
    for start in range(0, len(values), size): yield values[start:start + size]

def normalize_vectors(vectors):
    vectors = np.asarray(vectors, dtype='float32')
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)

def encode_huggingface(records, model_name, batch_size):
    import torch
    from transformers import AutoModel, AutoTokenizer
    tokenizer, model = AutoTokenizer.from_pretrained(model_name), AutoModel.from_pretrained(model_name)
    model.eval(); vectors = []
    with torch.no_grad():
        for batch in batches([r['search_text'] for r in records], batch_size):
            encoded = tokenizer(batch, padding=True, truncation=True, max_length=512, return_tensors='pt')
            output = model(**encoded).last_hidden_state
            mask = encoded['attention_mask'].unsqueeze(-1).float()
            vectors.append(((output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)).cpu().numpy())
    return normalize_vectors(np.concatenate(vectors, axis=0))

def encode_sentence_transformer(records, model_name, batch_size):
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer(model_name)
    vectors = model.encode([r['search_text'] for r in records], batch_size=batch_size, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True)
    del model; gc.collect()
    return np.asarray(vectors, dtype='float32')

def write_outputs(records, vectors, config):
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True); prefix = config['output']
    np.save(OUTPUT_DIR / f'{prefix}.npy', vectors)
    metadata = OUTPUT_DIR / f'{prefix}_metadata.jsonl'
    metadata.write_text(''.join(json.dumps({k: v for k, v in r.items() if k != 'search_text'}, ensure_ascii=False) + '\n' for r in records), encoding='utf-8')
    manifest = {'model': config['model'], 'provider': config['provider'], 'input_path': str(CHUNK_PATH), 'chunking_strategy': 'recursive_token_aware', 'max_tokens': MAX_TOKENS, 'tokenizer': TOKENIZER_NAME, 'min_chunk_chars': MIN_CHUNK_CHARS, 'context_fields': ['document_title', 'article_number', 'chapter_title', 'section_title'], 'chunk_count': len(records), 'embedding_dimension': int(vectors.shape[1]), 'normalized_embeddings': True}
    (OUTPUT_DIR / f'{prefix}_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print('Chunk input:', CHUNK_PATH)

In [ ]:
records = create_chunks()
results = {}
for model_key, config in MODELS.items():
    if config['provider'] == 'huggingface-mean-pooling':
        vectors = encode_huggingface(records, config['model'], config['batch_size'])
    else:
        vectors = encode_sentence_transformer(records, config['model'], config['batch_size'])
    write_outputs(records, vectors, config)
    results[model_key] = {'chunks': len(records), 'shape': list(vectors.shape)}
    del vectors; gc.collect()
print(json.dumps(results, ensure_ascii=False, indent=2))

In [ ]:
from sentence_transformers import SentenceTransformer
article_records = {}
for line in INPUT_PATH.read_text(encoding='utf-8').splitlines():
    article = json.loads(line)
    article_records.setdefault(article['article_id'], article)
article_records = list(article_records.values())
article_model = SentenceTransformer('BAAI/bge-m3')
article_texts = [embedding_text(article, article.get('search_text', article.get('text', ''))) for article in article_records]
article_vectors = article_model.encode(article_texts, batch_size=2, show_progress_bar=False, convert_to_numpy=True, normalize_embeddings=True).astype('float32')
np.save(OUTPUT_DIR / 'articles_bge_m3.npy', article_vectors)
(OUTPUT_DIR / 'articles_bge_m3_metadata.jsonl').write_text(''.join(json.dumps({'article_id': article['article_id'], 'document_id': article['document_id'], 'article_number': article['article_number']}, ensure_ascii=False) + '\n' for article in article_records), encoding='utf-8')
(OUTPUT_DIR / 'articles_bge_m3_manifest.json').write_text(json.dumps({'model': 'BAAI/bge-m3', 'input_path': str(INPUT_PATH), 'article_count': len(article_records), 'embedding_dimension': int(article_vectors.shape[1]), 'normalized_embeddings': True}, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
print('Article embeddings:', len(article_records), article_vectors.shape)